# Torus radius-shift simulation: power and validity

This notebook applies the Bonferroni, persistence-diagram (PD), and
persistence-landscape (PL) tests to
`npc=100_nset=500_torus_radius_pd.rds`.

- **Power:** for replication $b$, every method compares the first
  `SAMPLE_SIZE` diagrams in null block $b$ with the first `SAMPLE_SIZE`
  diagrams in shifted block $b$.
- **Validity:** the 500 null blocks are divided into 250 disjoint pairs:
  blocks $1{:}250$ are compared with blocks $251{:}500$. Each sample again
  contains the first `SAMPLE_SIZE` diagrams of its block.
- The Bonferroni test uses the linear persistence weight
  $w(b,d)=d-b$.
- The PD and PL tests use `B` permutations. Within a comparison, both
  permutation tests use the same reproducible seed.


## Dependencies

The notebook uses `rdata` to read the nested RDS object and imports the
three testing procedures from `TDA_Testing.py`. If `rdata` is unavailable,
install it once and restart the kernel:

```python
%pip install rdata
```


In [1]:
import gc
import hashlib
import inspect
import itertools
import os
import pickle
import random
import sys
import time
from collections.abc import Mapping
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from persim.landscapes import PersLandscapeApprox

import pickle
def savepkl(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
def loadpkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../../source"))
from TDA_Testing import *

try:
    import rdata
except ImportError as exc:
    raise ImportError(
        "Install the pure-Python RDS reader with `%pip install rdata`, "
        "then restart the kernel."
    ) from exc

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
RDS_PATH = Path("../../data/Rdata/npc=100_nset=500_torus_radius_combined.rds")
RESULT_DIR = Path(
    "../../results/simulation_results/Torus_results"
)
PL_CACHE_DIR = Path(
    "../../data/PLdata/torus_radius_shift_fixed_grid"
)

# This order must agree with the top-level list order in the RDS object.
RADIUS_SHIFTS = np.array([0.01, 0.016, 0.026, 0.036, 0.0])
NPC_STORED = 100
SAMPLE_SIZE = 50
NSET = 500
NUMBER_OF_NULL_PAIRS = NSET // 2
ALPHA = 0.05
B = 1000

MASTER_SEED = 42
MAX_BLOCK_ENTRIES = 1_000_000
PROGRESS_EVERY = 10

# All persistence landscapes use this one grid. 
PL_START = 0.0
PL_STOP = 3.0
PL_NUM_STEPS = 500
PL_CACHE_VERSION = 1

# False: reuse compatible cache filenames if they already exist.
# True: overwrite every shift-specific PL cache from the PD data.
REBUILD_PL_CACHE = False

RESULT_DIR.mkdir(parents=True, exist_ok=True)
PL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR.mkdir(parents=True, exist_ok=True)


BONFERRONI_SUMMARY_PATH = (
    RESULT_DIR / "torus_radius_bonferroni_summary.pkl"
)
PD_RESULT_PATH = RESULT_DIR / "torus_radius_pd_summary.pkl"
PL_RESULT_PATH = RESULT_DIR / "torus_radius_pl_summary.pkl"

VALIDITY_SUMMARY_PATH = RESULT_DIR / "torus_radius_validity_summary.pkl"

## Load and decode the RDS data

Each serialized diagram is the column-major flattening of a three-column
matrix containing homological dimension, birth, and death. The decoder
reconstructs that matrix and retains the $H_1$ birth-death pairs.


In [2]:
if not RDS_PATH.exists():
    raise FileNotFoundError(f"RDS file not found: {RDS_PATH.resolve()}")

load_start = time.time()

if hasattr(rdata, "read_rds"):
    torus_raw = rdata.read_rds(RDS_PATH)
else:
    parsed_rds = rdata.parser.parse_file(RDS_PATH)
    torus_raw = rdata.conversion.convert(parsed_rds)

print("RDS loaded in", f"{(time.time() - load_start) / 60:.3f} minutes")


RDS loaded in 0.151 minutes


In [3]:
def as_list(obj, label):
    """Convert a list-like object returned by rdata to a Python list."""
    if isinstance(obj, list):
        return obj
    if isinstance(obj, tuple):
        return list(obj)
    if isinstance(obj, Mapping):
        return list(obj.values())
    if isinstance(obj, np.ndarray) and obj.dtype == object:
        return list(obj.reshape(-1))
    raise TypeError(
        f"{label} should be list-like, but was converted to {type(obj)!r}."
    )


def decode_full_diagram(flat_diagram):
    """Reconstruct the matrix with columns (dimension, birth, death)."""
    flat = np.asarray(flat_diagram, dtype=float).reshape(-1)

    if flat.size % 3 != 0:
        raise ValueError(
            f"Serialized diagram length {flat.size} is not divisible by 3."
        )

    number_of_rows = flat.size // 3
    return np.ascontiguousarray(
        flat.reshape(3, number_of_rows).T,
        dtype=float,
    )


def decode_h1_diagram(flat_diagram):
    """Reconstruct one diagram and retain its H1 birth-death pairs."""
    full_diagram = decode_full_diagram(flat_diagram)
    h1 = full_diagram[
        np.isclose(full_diagram[:, 0], 1.0),
        1:3,
    ]

    if h1.size == 0:
        return np.empty((0, 2), dtype=float)

    if not np.all(np.isfinite(h1)):
        raise ValueError(
            "An H1 diagram contains a non-finite birth or death value. "
            "Check whether maxscale=3 truncates an essential interval."
        )

    return np.ascontiguousarray(h1, dtype=float)


def selected_raw_diagrams(raw_simulation):
    """Select diagrams 1:SAMPLE_SIZE from one simulation block."""
    raw_diagrams = as_list(raw_simulation, "one simulation block")

    if len(raw_diagrams) != NPC_STORED:
        raise ValueError(
            f"Expected {NPC_STORED} stored diagrams, "
            f"but found {len(raw_diagrams)}."
        )

    return raw_diagrams[:SAMPLE_SIZE]


def decode_simulation(raw_simulation):
    return [
        decode_h1_diagram(item)
        for item in selected_raw_diagrams(raw_simulation)
    ]


In [4]:
raw_shift_blocks = as_list(torus_raw, "top-level torus_pd")

if len(raw_shift_blocks) != len(RADIUS_SHIFTS):
    raise ValueError(
        f"Expected {len(RADIUS_SHIFTS)} radius-shift groups, "
        f"but found {len(raw_shift_blocks)}."
    )

simulation_blocks = []
for shift, raw_shift in zip(RADIUS_SHIFTS, raw_shift_blocks):
    blocks = as_list(raw_shift, f"radius_shift={shift:.3f}")
    if len(blocks) != NSET:
        raise ValueError(
            f"radius_shift={shift:.3f}: expected {NSET} simulation blocks, "
            f"but found {len(blocks)}."
        )
    simulation_blocks.append(blocks)

print(
    f"Validated {len(simulation_blocks)} shifts x {NSET} blocks x "
    f"{NPC_STORED} stored diagrams; the first {SAMPLE_SIZE} are used."
)

# Decode a few blocks to detect serialization and H1 errors early.
for shift_index, shift in enumerate(RADIUS_SHIFTS):
    h1_sizes = []
    for block_index in range(min(3, NSET)):
        h1_sizes.extend(
            len(diagram)
            for diagram in decode_simulation(
                simulation_blocks[shift_index][block_index]
            )
        )
    print(
        f"shift={shift:.3f}: H1 points per diagram "
        f"min/median/max={np.min(h1_sizes)}/"
        f"{np.median(h1_sizes):.0f}/{np.max(h1_sizes)}"
    )


Validated 5 shifts x 500 blocks x 100 stored diagrams; the first 50 are used.
shift=0.010: H1 points per diagram min/median/max=96/112/137
shift=0.016: H1 points per diagram min/median/max=93/113/132
shift=0.026: H1 points per diagram min/median/max=92/112/135
shift=0.036: H1 points per diagram min/median/max=91/114/136
shift=0.000: H1 points per diagram min/median/max=92/111/133


## Simulation design

Power replication $b$ uses null block $b$ and shifted block $b$.
Validity comparison $q$ uses null blocks $q$ and
$q+\texttt{NUMBER_OF_NULL_PAIRS}$. The validity pairs are therefore
disjoint. All three methods receive the same selected diagrams within
every power replication and every validity comparison.


In [5]:
null_candidates = np.flatnonzero(np.isclose(RADIUS_SHIFTS, 0.0))
if len(null_candidates) != 1:
    raise ValueError("RADIUS_SHIFTS must contain exactly one zero.")
if NSET % 2 != 0:
    raise ValueError("NSET must be even to construct disjoint null pairs.")

null_index = int(null_candidates[0])
alternative_indices = [
    index
    for index, shift in enumerate(RADIUS_SHIFTS)
    if shift > 0.0
]
alternative_shifts = RADIUS_SHIFTS[alternative_indices]

null_pair_blocks = np.column_stack((
    np.arange(NUMBER_OF_NULL_PAIRS, dtype=int),
    np.arange(NUMBER_OF_NULL_PAIRS, NSET, dtype=int),
))

seed_rng = np.random.default_rng(MASTER_SEED)
power_test_seeds = seed_rng.integers(
    1,
    np.iinfo(np.int32).max,
    size=(len(alternative_indices), NSET),
    dtype=np.int64,
)
validity_test_seeds = seed_rng.integers(
    1,
    np.iinfo(np.int32).max,
    size=NUMBER_OF_NULL_PAIRS,
    dtype=np.int64,
)

print(
    f"Power: {NSET} matched block comparisons per positive shift."
)
print(
    f"Validity: {NUMBER_OF_NULL_PAIRS} disjoint null-block pairs "
    f"({null_pair_blocks[0, 0] + 1} vs "
    f"{null_pair_blocks[0, 1] + 1}, ..., "
    f"{null_pair_blocks[-1, 0] + 1} vs "
    f"{null_pair_blocks[-1, 1] + 1})."
)


Power: 500 matched block comparisons per positive shift.
Validity: 250 disjoint null-block pairs (1 vs 251, ..., 250 vs 500).


## Bonferroni power

The persistence weight is $w(b,d)=d-b$. Each replication compares the
selected null and shifted diagrams from the same block number.


In [39]:
func_weight = function_weight("Poly", poly_order=2)

test_kwargs = {
    "alpha": ALPHA,
    "func_weight": func_weight,
    "reference_rng":np.random.default_rng(20260928),
    "max_block_entries": MAX_BLOCK_ENTRIES,
}


In [40]:
bonferroni_rejections = np.zeros(
    (len(alternative_indices), NSET),
    dtype=bool,
)
bonferroni_pvalues = np.full(
    (len(alternative_indices), NSET),
    np.nan,
    dtype=float,
)

start_time = time.time()

for replication in range(NSET):
    null_sample = decode_simulation(
        simulation_blocks[null_index][replication]
    )

    for output_index, shift_index in enumerate(alternative_indices):
        shifted_sample = decode_simulation(
            simulation_blocks[shift_index][replication]
        )
        result = bonferroni_test(
            null_sample,
            shifted_sample,
            **test_kwargs,
        )
        bonferroni_rejections[output_index, replication] = bool(
            result["reject"]
        )
        bonferroni_pvalues[output_index, replication] = float(
            result.get("adjusted_p_value", np.nan)
        )

    completed = replication + 1
    if completed % PROGRESS_EVERY == 0 or completed == NSET:
        elapsed_minutes = (time.time() - start_time) / 60.0
        current_power = bonferroni_rejections[:, :completed].mean(axis=1)
        power_text = ", ".join(
            f"shift={shift:.3f}: {power:.3f}"
            for shift, power in zip(alternative_shifts, current_power)
        )
        print(
            f"Bonferroni power {completed}/{NSET} completed | "
            f"{power_text} | elapsed={elapsed_minutes:.1f} min"
        )


Bonferroni power 10/500 completed | shift=0.010: 0.400, shift=0.016: 0.700, shift=0.026: 1.000, shift=0.036: 1.000 | elapsed=6.8 min
Bonferroni power 20/500 completed | shift=0.010: 0.350, shift=0.016: 0.650, shift=0.026: 1.000, shift=0.036: 1.000 | elapsed=13.7 min
Bonferroni power 30/500 completed | shift=0.010: 0.333, shift=0.016: 0.633, shift=0.026: 1.000, shift=0.036: 1.000 | elapsed=20.5 min
Bonferroni power 40/500 completed | shift=0.010: 0.300, shift=0.016: 0.625, shift=0.026: 1.000, shift=0.036: 1.000 | elapsed=27.2 min
Bonferroni power 50/500 completed | shift=0.010: 0.300, shift=0.016: 0.660, shift=0.026: 1.000, shift=0.036: 1.000 | elapsed=34.0 min
Bonferroni power 60/500 completed | shift=0.010: 0.267, shift=0.016: 0.633, shift=0.026: 1.000, shift=0.036: 1.000 | elapsed=40.8 min
Bonferroni power 70/500 completed | shift=0.010: 0.271, shift=0.016: 0.629, shift=0.026: 1.000, shift=0.036: 1.000 | elapsed=47.6 min
Bonferroni power 80/500 completed | shift=0.010: 0.263, shift=0

In [41]:
bonferroni_frames = []
for output_index, shift in enumerate(alternative_shifts):
    bonferroni_frames.append(
        pd.DataFrame({
            "method": "Bonferroni",
            "radius_shift": float(shift),
            "replication": np.arange(1, NSET + 1),
            "reject": bonferroni_rejections[output_index].astype(int),
            "p_value": bonferroni_pvalues[output_index],
            "p_value_type": "Bonferroni-adjusted Gaussian",
        })
    )

bonferroni_results = pd.concat(bonferroni_frames, ignore_index=True)

bonferroni_summary = (
    bonferroni_results
    .groupby(["method", "radius_shift"], as_index=False)
    .agg(
        replications=("reject", "size"),
        rejections=("reject", "sum"),
        rejection_rate=("reject", "mean"),
    )
)
bonferroni_summary["mcse"] = np.sqrt(
    bonferroni_summary["rejection_rate"]
    * (1.0 - bonferroni_summary["rejection_rate"])
    / bonferroni_summary["replications"]
)
bonferroni_summary["ci_95_lower"] = np.maximum(
    0.0,
    bonferroni_summary["rejection_rate"]
    - 1.96 * bonferroni_summary["mcse"],
)
bonferroni_summary["ci_95_upper"] = np.minimum(
    1.0,
    bonferroni_summary["rejection_rate"]
    + 1.96 * bonferroni_summary["mcse"],
)
bonferroni_summary["n"] = SAMPLE_SIZE
bonferroni_summary["m"] = SAMPLE_SIZE
bonferroni_summary["alpha"] = ALPHA
# bonferroni_summary["test_module_sha256"] = test_module_hash
savepkl(bonferroni_summary, RESULT_DIR / "torus_radius_bonferroni_square_summary.pkl")

bonferroni_summary

,method,radius_shift,replications,rejections,rejection_rate,mcse,ci_95_lower,ci_95_upper,n,m,alpha
0,Bonferroni,0.010,500,128,0.256,0.019517,0.217746,0.294254,50,50,0.05
1,Bonferroni,0.016,500,336,0.672,0.020996,0.630848,0.713152,50,50,0.05
2,Bonferroni,0.026,500,500,1.000,0.000000,1.000000,1.000000,50,50,0.05
3,Bonferroni,0.036,500,500,1.000,0.000000,1.000000,1.000000,50,50,0.05


## Linear weight

In [42]:
func_weight = function_weight("Poly", poly_order=1)
test_kwargs = {
    "alpha": ALPHA,
    "func_weight": func_weight,
    "reference_rng":np.random.default_rng(20260928),
    "max_block_entries": MAX_BLOCK_ENTRIES,
}

bonferroni_rejections = np.zeros(
    (len(alternative_indices), NSET),
    dtype=bool,
)
bonferroni_pvalues = np.full(
    (len(alternative_indices), NSET),
    np.nan,
    dtype=float,
)

start_time = time.time()

for replication in range(NSET):
    null_sample = decode_simulation(
        simulation_blocks[null_index][replication]
    )

    for output_index, shift_index in enumerate(alternative_indices):
        shifted_sample = decode_simulation(
            simulation_blocks[shift_index][replication]
        )
        result = bonferroni_test(
            null_sample,
            shifted_sample,
            **test_kwargs,
        )
        bonferroni_rejections[output_index, replication] = bool(
            result["reject"]
        )
        bonferroni_pvalues[output_index, replication] = float(
            result.get("adjusted_p_value", np.nan)
        )

    completed = replication + 1
    if completed % PROGRESS_EVERY == 0 or completed == NSET:
        elapsed_minutes = (time.time() - start_time) / 60.0
        current_power = bonferroni_rejections[:, :completed].mean(axis=1)
        power_text = ", ".join(
            f"shift={shift:.3f}: {power:.3f}"
            for shift, power in zip(alternative_shifts, current_power)
        )
        print(
            f"Bonferroni power {completed}/{NSET} completed | "
            f"{power_text} | elapsed={elapsed_minutes:.1f} min"
        )
bonferroni_frames = []

for output_index, shift in enumerate(alternative_shifts):
    bonferroni_frames.append(
        pd.DataFrame({
            "method": "Bonferroni",
            "radius_shift": float(shift),
            "replication": np.arange(1, NSET + 1),
            "reject": bonferroni_rejections[output_index].astype(int),
            "p_value": bonferroni_pvalues[output_index],
            "p_value_type": "Bonferroni-adjusted Gaussian",
        })
    )

bonferroni_results = pd.concat(bonferroni_frames, ignore_index=True)

bonferroni_summary = (
    bonferroni_results
    .groupby(["method", "radius_shift"], as_index=False)
    .agg(
        replications=("reject", "size"),
        rejections=("reject", "sum"),
        rejection_rate=("reject", "mean"),
    )
)
bonferroni_summary["mcse"] = np.sqrt(
    bonferroni_summary["rejection_rate"]
    * (1.0 - bonferroni_summary["rejection_rate"])
    / bonferroni_summary["replications"]
)
bonferroni_summary["ci_95_lower"] = np.maximum(
    0.0,
    bonferroni_summary["rejection_rate"]
    - 1.96 * bonferroni_summary["mcse"],
)
bonferroni_summary["ci_95_upper"] = np.minimum(
    1.0,
    bonferroni_summary["rejection_rate"]
    + 1.96 * bonferroni_summary["mcse"],
)
bonferroni_summary["n"] = SAMPLE_SIZE
bonferroni_summary["m"] = SAMPLE_SIZE
bonferroni_summary["alpha"] = ALPHA
# bonferroni_summary["test_module_sha256"] = test_module_hash
savepkl(bonferroni_summary, RESULT_DIR / "torus_radius_bonferroni_linear_summary.pkl")

bonferroni_summary

Bonferroni power 10/500 completed | shift=0.010: 0.200, shift=0.016: 0.400, shift=0.026: 0.900, shift=0.036: 1.000 | elapsed=6.8 min
Bonferroni power 20/500 completed | shift=0.010: 0.250, shift=0.016: 0.450, shift=0.026: 0.900, shift=0.036: 1.000 | elapsed=13.6 min
Bonferroni power 30/500 completed | shift=0.010: 0.267, shift=0.016: 0.467, shift=0.026: 0.900, shift=0.036: 1.000 | elapsed=20.3 min
Bonferroni power 40/500 completed | shift=0.010: 0.225, shift=0.016: 0.450, shift=0.026: 0.900, shift=0.036: 1.000 | elapsed=27.1 min
Bonferroni power 50/500 completed | shift=0.010: 0.260, shift=0.016: 0.460, shift=0.026: 0.920, shift=0.036: 1.000 | elapsed=33.8 min
Bonferroni power 60/500 completed | shift=0.010: 0.250, shift=0.016: 0.433, shift=0.026: 0.933, shift=0.036: 1.000 | elapsed=40.6 min
Bonferroni power 70/500 completed | shift=0.010: 0.243, shift=0.016: 0.414, shift=0.026: 0.929, shift=0.036: 1.000 | elapsed=47.4 min
Bonferroni power 80/500 completed | shift=0.010: 0.237, shift=0

,method,radius_shift,replications,rejections,rejection_rate,mcse,ci_95_lower,ci_95_upper,n,m,alpha
0,Bonferroni,0.010,500,99,0.198,0.017821,0.163071,0.232929,50,50,0.05
1,Bonferroni,0.016,500,234,0.468,0.022315,0.424263,0.511737,50,50,0.05
2,Bonferroni,0.026,500,479,0.958,0.008971,0.940418,0.975582,50,50,0.05
3,Bonferroni,0.036,500,500,1.000,0.000000,1.000000,1.000000,50,50,0.05


## Constant 

In [43]:
func_weight = function_weight("constant")
test_kwargs = {
    "alpha": ALPHA,
    "func_weight": func_weight,
    "reference_rng":np.random.default_rng(20260928),
    "max_block_entries": MAX_BLOCK_ENTRIES,
}

bonferroni_rejections = np.zeros(
    (len(alternative_indices), NSET),
    dtype=bool,
)
bonferroni_pvalues = np.full(
    (len(alternative_indices), NSET),
    np.nan,
    dtype=float,
)

start_time = time.time()

for replication in range(NSET):
    null_sample = decode_simulation(
        simulation_blocks[null_index][replication]
    )

    for output_index, shift_index in enumerate(alternative_indices):
        shifted_sample = decode_simulation(
            simulation_blocks[shift_index][replication]
        )
        result = bonferroni_test(
            null_sample,
            shifted_sample,
            **test_kwargs,
        )
        bonferroni_rejections[output_index, replication] = bool(
            result["reject"]
        )
        bonferroni_pvalues[output_index, replication] = float(
            result.get("adjusted_p_value", np.nan)
        )

    completed = replication + 1
    if completed % PROGRESS_EVERY == 0 or completed == NSET:
        elapsed_minutes = (time.time() - start_time) / 60.0
        current_power = bonferroni_rejections[:, :completed].mean(axis=1)
        power_text = ", ".join(
            f"shift={shift:.3f}: {power:.3f}"
            for shift, power in zip(alternative_shifts, current_power)
        )
        print(
            f"Bonferroni power {completed}/{NSET} completed | "
            f"{power_text} | elapsed={elapsed_minutes:.1f} min"
        )
bonferroni_frames = []

for output_index, shift in enumerate(alternative_shifts):
    bonferroni_frames.append(
        pd.DataFrame({
            "method": "Bonferroni",
            "radius_shift": float(shift),
            "replication": np.arange(1, NSET + 1),
            "reject": bonferroni_rejections[output_index].astype(int),
            "p_value": bonferroni_pvalues[output_index],
            "p_value_type": "Bonferroni-adjusted Gaussian",
        })
    )

bonferroni_results = pd.concat(bonferroni_frames, ignore_index=True)

bonferroni_summary = (
    bonferroni_results
    .groupby(["method", "radius_shift"], as_index=False)
    .agg(
        replications=("reject", "size"),
        rejections=("reject", "sum"),
        rejection_rate=("reject", "mean"),
    )
)
bonferroni_summary["mcse"] = np.sqrt(
    bonferroni_summary["rejection_rate"]
    * (1.0 - bonferroni_summary["rejection_rate"])
    / bonferroni_summary["replications"]
)
bonferroni_summary["ci_95_lower"] = np.maximum(
    0.0,
    bonferroni_summary["rejection_rate"]
    - 1.96 * bonferroni_summary["mcse"],
)
bonferroni_summary["ci_95_upper"] = np.minimum(
    1.0,
    bonferroni_summary["rejection_rate"]
    + 1.96 * bonferroni_summary["mcse"],
)
bonferroni_summary["n"] = SAMPLE_SIZE
bonferroni_summary["m"] = SAMPLE_SIZE
bonferroni_summary["alpha"] = ALPHA
# bonferroni_summary["test_module_sha256"] = test_module_hash
savepkl(bonferroni_summary, RESULT_DIR / "torus_radius_bonferroni_constant_summary.pkl")

bonferroni_summary

Bonferroni power 10/500 completed | shift=0.010: 0.000, shift=0.016: 0.000, shift=0.026: 0.100, shift=0.036: 0.100 | elapsed=6.8 min
Bonferroni power 20/500 completed | shift=0.010: 0.000, shift=0.016: 0.050, shift=0.026: 0.100, shift=0.036: 0.100 | elapsed=13.6 min
Bonferroni power 30/500 completed | shift=0.010: 0.067, shift=0.016: 0.067, shift=0.026: 0.133, shift=0.036: 0.067 | elapsed=20.4 min
Bonferroni power 40/500 completed | shift=0.010: 0.050, shift=0.016: 0.050, shift=0.026: 0.125, shift=0.036: 0.050 | elapsed=27.2 min
Bonferroni power 50/500 completed | shift=0.010: 0.040, shift=0.016: 0.060, shift=0.026: 0.100, shift=0.036: 0.040 | elapsed=33.9 min
Bonferroni power 60/500 completed | shift=0.010: 0.050, shift=0.016: 0.067, shift=0.026: 0.083, shift=0.036: 0.067 | elapsed=40.7 min
Bonferroni power 70/500 completed | shift=0.010: 0.043, shift=0.016: 0.057, shift=0.026: 0.071, shift=0.036: 0.071 | elapsed=47.5 min
Bonferroni power 80/500 completed | shift=0.010: 0.050, shift=0

,method,radius_shift,replications,rejections,rejection_rate,mcse,ci_95_lower,ci_95_upper,n,m,alpha
0,Bonferroni,0.010,500,32,0.064,0.010946,0.042546,0.085454,50,50,0.05
1,Bonferroni,0.016,500,33,0.066,0.011104,0.044237,0.087763,50,50,0.05
2,Bonferroni,0.026,500,34,0.068,0.011258,0.045933,0.090067,50,50,0.05
3,Bonferroni,0.036,500,41,0.082,0.012270,0.057951,0.106049,50,50,0.05


## PD power

This is the Robinson--Turner persistence-diagram permutation test. Each
call receives the same null and shifted diagrams used by the Bonferroni
test. The replication-specific seed is passed directly to
`permutation_test`.


In [31]:
pd_pvalues = np.full(
    (len(alternative_indices), NSET),
    np.nan,
    dtype=float,
)

start_time = time.time()

for replication in range(NSET):
    null_sample = decode_simulation(
        simulation_blocks[null_index][replication]
    )

    for output_index, shift_index in enumerate(alternative_indices):
        shifted_sample = decode_simulation(
            simulation_blocks[shift_index][replication]
        )
        seed_j = int(power_test_seeds[output_index, replication])
        _, _, pd_pvalues[output_index, replication] = permutation_test(
            null_sample,
            shifted_sample,
            num_permutations=B,
            seed=seed_j,
        )

    completed = replication + 1
    if completed % PROGRESS_EVERY == 0 or completed == NSET:
        elapsed_minutes = (time.time() - start_time) / 60.0
        current_power = np.mean(
            pd_pvalues[:, :completed] <= ALPHA,
            axis=1,
        )
        power_text = ", ".join(
            f"shift={shift:.3f}: {power:.3f}"
            for shift, power in zip(alternative_shifts, current_power)
        )
        print(
            f"PD power {completed}/{NSET} completed | {power_text} | "
            f"elapsed={elapsed_minutes:.1f} min"
        )

pd_frames = []
for output_index, shift in enumerate(alternative_shifts):
    pd_frames.append(
        pd.DataFrame({
            "method": "PD",
            "radius_shift": float(shift),
            "replication": np.arange(1, NSET + 1),
            "reject": (pd_pvalues[output_index] <= ALPHA).astype(int),
            "p_value": pd_pvalues[output_index],
            "p_value_type": f"permutation (B={B})",
        })
    )

pd_results = pd.concat(pd_frames, ignore_index=True)
pd_summary = (
    pd_results
    .groupby(["method", "radius_shift"], as_index=False)
    .agg(
        replications=("reject", "size"),
        rejections=("reject", "sum"),
        rejection_rate=("reject", "mean"),
    )
)
savepkl(pd_summary,PD_RESULT_PATH)

PD power 10/500 completed | shift=0.010: 0.100, shift=0.016: 0.000, shift=0.026: 0.200, shift=0.036: 0.600 | elapsed=3.9 min
PD power 20/500 completed | shift=0.010: 0.100, shift=0.016: 0.150, shift=0.026: 0.100, shift=0.036: 0.600 | elapsed=7.8 min
PD power 30/500 completed | shift=0.010: 0.100, shift=0.016: 0.133, shift=0.026: 0.167, shift=0.036: 0.600 | elapsed=11.7 min
PD power 40/500 completed | shift=0.010: 0.100, shift=0.016: 0.125, shift=0.026: 0.175, shift=0.036: 0.625 | elapsed=15.5 min
PD power 50/500 completed | shift=0.010: 0.080, shift=0.016: 0.100, shift=0.026: 0.180, shift=0.036: 0.640 | elapsed=19.4 min
PD power 60/500 completed | shift=0.010: 0.067, shift=0.016: 0.117, shift=0.026: 0.200, shift=0.036: 0.617 | elapsed=23.3 min
PD power 70/500 completed | shift=0.010: 0.071, shift=0.016: 0.100, shift=0.026: 0.200, shift=0.036: 0.614 | elapsed=27.2 min
PD power 80/500 completed | shift=0.010: 0.075, shift=0.016: 0.087, shift=0.026: 0.200, shift=0.036: 0.613 | elapsed=31.

In [32]:
import random
import numpy as np
from persim.landscapes import PersLandscapeApprox

PL_START = 0.0
PL_STOP = 3.0       # ripsDiag의 maxscale과 동일
PL_NUM_STEPS = 500


def persistence_landscape_from_h1(h1_diagram):
    h1_diagram = np.asarray(h1_diagram, dtype=float).reshape(-1, 2)

    if h1_diagram.size == 0:
        return PersLandscapeApprox(
            start=PL_START,
            stop=PL_STOP,
            num_steps=PL_NUM_STEPS,
            hom_deg=1,
            values=np.zeros((1, PL_NUM_STEPS), dtype=float),
        )

    if (
        np.min(h1_diagram[:, 0]) < PL_START
        or np.max(h1_diagram[:, 1]) > PL_STOP
    ):
        raise ValueError("A persistence point lies outside the PL grid.")

    diagrams_by_degree = [
        np.empty((0, 2), dtype=float),
        h1_diagram,
    ]

    return PersLandscapeApprox(
        dgms=diagrams_by_degree,
        hom_deg=1,
        start=PL_START,
        stop=PL_STOP,
        num_steps=PL_NUM_STEPS,
    )


def landscape_simulation(raw_simulation):
    return [
        persistence_landscape_from_h1(decode_h1_diagram(item))
        for item in selected_raw_diagrams(raw_simulation)
    ]

## PL preprocessing and power

The selected $H_1$ diagrams are first converted to approximate persistence
landscapes on the common grid `[PL_START, PL_STOP]` with
`PL_NUM_STEPS` grid points. This conversion is deterministic and does not
involve the permutation seed.

One streamed pickle is written per radius shift. The first pickle record is
metadata, followed by `NSET` records with structure
`[diagram][landscape_depth, PL_NUM_STEPS]`. Each simulation block can
therefore be read independently and discarded after testing. Both power
and validity consume these same cached records; they never reconstruct a
landscape from its PD.


In [33]:
def persistence_landscape_values_from_h1(h1_diagram):
    """Return the fixed-grid H1 landscape values for one PD."""
    h1_diagram = np.asarray(h1_diagram, dtype=float)

    if h1_diagram.size == 0:
        # The landscape of an empty diagram is the zero function.
        return np.zeros((1, PL_NUM_STEPS), dtype=float)

    if h1_diagram.ndim != 2 or h1_diagram.shape[1] != 2:
        raise ValueError("An H1 diagram must have shape (points, 2).")
    if not np.all(np.isfinite(h1_diagram)):
        raise ValueError("An H1 diagram contains a non-finite value.")
    if np.any(h1_diagram[:, 1] < h1_diagram[:, 0]):
        raise ValueError("An H1 interval has death smaller than birth.")

    grid_tolerance = 64.0 * np.finfo(float).eps * max(
        1.0,
        abs(PL_START),
        abs(PL_STOP),
    )
    if (
        np.min(h1_diagram[:, 0]) < PL_START - grid_tolerance
        or np.max(h1_diagram[:, 1]) > PL_STOP + grid_tolerance
    ):
        raise ValueError(
            "A birth-death pair lies outside the common PL grid "
            f"[{PL_START}, {PL_STOP}]."
        )

    diagrams_by_degree = [
        np.empty((0, 2), dtype=float),
        np.ascontiguousarray(h1_diagram, dtype=float),
    ]
    landscape = PersLandscapeApprox(
        dgms=diagrams_by_degree,
        hom_deg=1,
        start=PL_START,
        stop=PL_STOP,
        num_steps=PL_NUM_STEPS,
    )

    # Very short intervals can all collapse to zero after grid snapping.
    # In that case some persim versions use a non-numeric "empty" marker;
    # the correct sampled landscape is simply zero.
    try:
        values = np.asarray(landscape.values, dtype=float)
    except (TypeError, ValueError):
        values = np.zeros((1, PL_NUM_STEPS), dtype=float)

    if values.size == 0:
        values = np.zeros((1, PL_NUM_STEPS), dtype=float)
    if values.ndim == 1 and values.size == PL_NUM_STEPS:
        values = values.reshape(1, -1)
    if values.ndim != 2 or values.shape[1] != PL_NUM_STEPS:
        raise ValueError(
            "Unexpected persistence-landscape value shape: "
            f"{values.shape}."
        )
    if not np.all(np.isfinite(values)):
        raise ValueError("A persistence landscape contains non-finite values.")

    return np.ascontiguousarray(values, dtype=float)


def landscape_value_simulation(raw_simulation):
    """Convert the selected diagrams in one block to fixed-grid PL values."""
    return [
        persistence_landscape_values_from_h1(
            decode_h1_diagram(flat_diagram)
        )
        for flat_diagram in selected_raw_diagrams(raw_simulation)
    ]


def _number_token(value):
    return f"{float(value):.4f}".replace("-", "m").replace(".", "p")


def pl_cache_path(radius_shift):
    grid_text = (
        f"{_number_token(PL_START)}_"
        f"{_number_token(PL_STOP)}_"
        f"{PL_NUM_STEPS}"
    )
    return PL_CACHE_DIR / (
        f"torus_pl_shift_{_number_token(radius_shift)}_"
        f"n{SAMPLE_SIZE}_nset{NSET}_grid_{grid_text}.pkl"
    )


def expected_pl_cache_metadata(radius_shift):
    return {
        "cache_version": PL_CACHE_VERSION,
        "file_format": "metadata_then_nset_block_pickles",
        "representation": "PersLandscapeApprox.values",
        "rds_filename": RDS_PATH.name,
        "rds_size_bytes": RDS_PATH.stat().st_size,
        "radius_shift": float(radius_shift),
        "npc_stored": NPC_STORED,
        "sample_size": SAMPLE_SIZE,
        "nset": NSET,
        "homological_degree": 1,
        "pl_start": float(PL_START),
        "pl_stop": float(PL_STOP),
        "pl_num_steps": PL_NUM_STEPS,
    }


def _validate_pl_cache_metadata(radius_shift, observed, path):
    expected = expected_pl_cache_metadata(radius_shift)
    mismatches = {
        key: (observed.get(key), expected_value)
        for key, expected_value in expected.items()
        if observed.get(key) != expected_value
    }
    if mismatches:
        raise ValueError(
            f"PL cache metadata in {path} does not match the current "
            f"settings: {mismatches}. Set REBUILD_PL_CACHE=True and "
            "rerun the preprocessing cell."
        )


def _validate_pl_block(block, replication, path):
    if not isinstance(block, list) or len(block) != SAMPLE_SIZE:
        raise ValueError(
            f"PL cache block {replication + 1} in {path} does not "
            f"contain {SAMPLE_SIZE} diagrams."
        )

    for diagram_index, values in enumerate(block):
        values = np.asarray(values)
        if values.ndim != 2 or values.shape[1] != PL_NUM_STEPS:
            raise ValueError(
                f"Invalid PL shape {values.shape} in block "
                f"{replication + 1}, diagram {diagram_index + 1}."
            )
        if not np.all(np.isfinite(values)):
            raise ValueError(
                f"Non-finite PL value in block {replication + 1}, "
                f"diagram {diagram_index + 1}."
            )

    return block


def validate_pl_cache_header(radius_shift):
    """Validate one cache without loading its landscape blocks."""
    path = pl_cache_path(radius_shift)
    if not path.exists():
        raise FileNotFoundError(f"PL cache not found: {path.resolve()}")

    try:
        with path.open("rb") as handle:
            metadata = pickle.load(handle)
    except (EOFError, pickle.UnpicklingError) as exc:
        raise ValueError(f"Cannot read PL cache header: {path}") from exc

    if not isinstance(metadata, dict):
        raise ValueError(f"Invalid PL cache metadata in {path}.")
    _validate_pl_cache_metadata(radius_shift, metadata, path)
    return path


def iter_pl_cache(radius_shift):
    """Yield one validated simulation block at a time from a PL cache."""
    path = pl_cache_path(radius_shift)
    if not path.exists():
        raise FileNotFoundError(
            f"PL cache not found: {path.resolve()}. "
            "Run the PL preprocessing cell first."
        )

    with path.open("rb") as handle:
        try:
            metadata = pickle.load(handle)
        except (EOFError, pickle.UnpicklingError) as exc:
            raise ValueError(f"Cannot read PL cache header: {path}") from exc

        if not isinstance(metadata, dict):
            raise ValueError(f"Invalid PL cache metadata in {path}.")
        _validate_pl_cache_metadata(radius_shift, metadata, path)

        for replication in range(NSET):
            try:
                block = pickle.load(handle)
            except EOFError as exc:
                raise ValueError(
                    f"PL cache {path} ended before block "
                    f"{replication + 1}/{NSET}. Rebuild it."
                ) from exc
            except pickle.UnpicklingError as exc:
                raise ValueError(
                    f"Cannot decode block {replication + 1} in {path}."
                ) from exc

            yield _validate_pl_block(block, replication, path)

        # A compatible file contains exactly one header and NSET blocks.
        try:
            pickle.load(handle)
        except EOFError:
            return
        raise ValueError(
            f"PL cache {path} contains records beyond the expected "
            f"{NSET} blocks. Rebuild it."
        )


def load_pl_cache_block(radius_shift, block_index):
    """Load one block while closing the sequential cache stream promptly."""
    if not (0 <= block_index < NSET):
        raise IndexError(f"block_index must lie in [0, {NSET - 1}].")

    stream = iter_pl_cache(radius_shift)
    try:
        return next(itertools.islice(stream, block_index, None))
    finally:
        stream.close()


### Build the shift-specific PL caches

On the first run this cell performs all PD-to-PL conversions. Each block is
written immediately, so preprocessing does not accumulate a full shift in
RAM. On later runs, an existing cache header is validated and reused. Set
`REBUILD_PL_CACHE=True` only when the underlying RDS data were replaced or
when the conversion itself must be rerun.


In [34]:
pl_cache_paths = {}

for shift_index, radius_shift in enumerate(RADIUS_SHIFTS):
    cache_path = pl_cache_path(radius_shift)
    pl_cache_paths[shift_index] = cache_path

    if cache_path.exists() and not REBUILD_PL_CACHE:
        validate_pl_cache_header(radius_shift)
        print(
            f"Using existing PL cache for shift={radius_shift:.3f}: "
            f"{cache_path.resolve()}"
        )
        continue

    shift_start = time.time()
    temporary_path = cache_path.with_suffix(cache_path.suffix + ".tmp")

    try:
        with temporary_path.open("wb") as handle:
            pickle.dump(
                expected_pl_cache_metadata(radius_shift),
                handle,
                protocol=pickle.HIGHEST_PROTOCOL,
            )

            for replication in range(NSET):
                block = landscape_value_simulation(
                    simulation_blocks[shift_index][replication]
                )
                pickle.dump(
                    block,
                    handle,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )

                completed = replication + 1
                if (
                    completed % PROGRESS_EVERY == 0
                    or completed == NSET
                ):
                    elapsed_minutes = (
                        time.time() - shift_start
                    ) / 60.0
                    print(
                        f"PL conversion shift={radius_shift:.3f}: "
                        f"{completed}/{NSET} blocks | "
                        f"elapsed={elapsed_minutes:.1f} min"
                    )

                del block

        os.replace(temporary_path, cache_path)
    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    cache_size_mb = cache_path.stat().st_size / (1024.0 ** 2)
    print(
        f"Saved shift={radius_shift:.3f} PL cache: "
        f"{cache_path.resolve()} ({cache_size_mb:.1f} MB)"
    )
    gc.collect()


Using existing PL cache for shift=0.010: /Users/youngunghan/Desktop/TDA-github-upload/data/PLdata/torus_radius_shift_fixed_grid/torus_pl_shift_0p0100_n50_nset500_grid_0p0000_3p0000_500.pkl
Using existing PL cache for shift=0.016: /Users/youngunghan/Desktop/TDA-github-upload/data/PLdata/torus_radius_shift_fixed_grid/torus_pl_shift_0p0160_n50_nset500_grid_0p0000_3p0000_500.pkl
Using existing PL cache for shift=0.026: /Users/youngunghan/Desktop/TDA-github-upload/data/PLdata/torus_radius_shift_fixed_grid/torus_pl_shift_0p0260_n50_nset500_grid_0p0000_3p0000_500.pkl
Using existing PL cache for shift=0.036: /Users/youngunghan/Desktop/TDA-github-upload/data/PLdata/torus_radius_shift_fixed_grid/torus_pl_shift_0p0360_n50_nset500_grid_0p0000_3p0000_500.pkl
Using existing PL cache for shift=0.000: /Users/youngunghan/Desktop/TDA-github-upload/data/PLdata/torus_radius_shift_fixed_grid/torus_pl_shift_0p0000_n50_nset500_grid_0p0000_3p0000_500.pkl


In [35]:
#2. PL POWER
pl_pvalues = np.full(
    (len(alternative_indices), NSET),
    np.nan,
    dtype=float,
)

# For each alternative shift, stream the matching null and alternative
# blocks together. Only two simulation blocks are resident at a time.
pl_power_start = time.time()

for output_index, shift_index in enumerate(alternative_indices):
    radius_shift = RADIUS_SHIFTS[shift_index]
    null_stream = iter_pl_cache(RADIUS_SHIFTS[null_index])
    shifted_stream = iter_pl_cache(radius_shift)
    shift_start = time.time()
    completed = 0

    try:
        for replication, (null_sample, shifted_sample) in enumerate(
            zip(null_stream, shifted_stream)
        ):
            seed_j = int(power_test_seeds[output_index, replication])
            pl_pvalues[
                output_index,
                replication,
            ] = permutation_pl_test_fast(
                null_sample,
                shifted_sample,
                seed=seed_j,
                num_perms=B,
            )

            completed = replication + 1
            if completed % PROGRESS_EVERY == 0 or completed == NSET:
                elapsed_shift_minutes = (
                    time.time() - shift_start
                ) / 60.0
                current_power = np.mean(
                    pl_pvalues[output_index, :completed] <= ALPHA
                )
                print(
                    f"PL power shift={radius_shift:.3f}: "
                    f"{completed}/{NSET} completed | "
                    f"power={current_power:.3f} | "
                    f"elapsed for shift={elapsed_shift_minutes:.1f} min"
                )
    finally:
        null_stream.close()
        shifted_stream.close()

    if completed != NSET:
        raise RuntimeError(
            f"Expected {NSET} PL power comparisons for "
            f"shift={radius_shift:.3f}, but completed {completed}."
        )
    gc.collect()

print(
    "Total PL power time:",
    f"{(time.time() - pl_power_start) / 60.0:.1f} min",
)

pl_frames = []
for output_index, shift in enumerate(alternative_shifts):
    pl_frames.append(
        pd.DataFrame({
            "method": "PL",
            "radius_shift": float(shift),
            "replication": np.arange(1, NSET + 1),
            "reject": (pl_pvalues[output_index] <= ALPHA).astype(int),
            "p_value": pl_pvalues[output_index],
            "p_value_type": (
                f"permutation (B={B}; fixed-grid PL Gram)"
            ),
        })
    )

pl_results = pd.concat(pl_frames, ignore_index=True)
PL_result = (
    pl_results
    .groupby(["method", "radius_shift"], as_index=False)
    .agg(
        replications=("reject", "size"),
        rejections=("reject", "sum"),
        rejection_rate=("reject", "mean"),
    )
)

savepkl(PL_result,PL_RESULT_PATH)
print("Saved PL results to:", PL_RESULT_PATH.resolve())


PL power shift=0.010: 10/500 completed | power=0.000 | elapsed for shift=0.0 min
PL power shift=0.010: 20/500 completed | power=0.150 | elapsed for shift=0.0 min
PL power shift=0.010: 30/500 completed | power=0.133 | elapsed for shift=0.1 min
PL power shift=0.010: 40/500 completed | power=0.175 | elapsed for shift=0.1 min
PL power shift=0.010: 50/500 completed | power=0.160 | elapsed for shift=0.1 min
PL power shift=0.010: 60/500 completed | power=0.167 | elapsed for shift=0.1 min
PL power shift=0.010: 70/500 completed | power=0.143 | elapsed for shift=0.2 min
PL power shift=0.010: 80/500 completed | power=0.125 | elapsed for shift=0.2 min
PL power shift=0.010: 90/500 completed | power=0.111 | elapsed for shift=0.2 min
PL power shift=0.010: 100/500 completed | power=0.110 | elapsed for shift=0.2 min
PL power shift=0.010: 110/500 completed | power=0.100 | elapsed for shift=0.3 min
PL power shift=0.010: 120/500 completed | power=0.108 | elapsed for shift=0.3 min
PL power shift=0.010: 130

In [36]:
PL_result

,method,radius_shift,replications,rejections,rejection_rate
0,PL,0.010,500,70,0.140
1,PL,0.016,500,214,0.428
2,PL,0.026,500,491,0.982
3,PL,0.036,500,500,1.000


## Validity under the null model

For validity comparison $q=1,\ldots,250$, sample $X_q$ consists of the
first 50 diagrams in null block $q$, while sample $Y_q$ consists of the
first 50 diagrams in null block $q+250$. Thus all 500 null blocks are used
exactly once and no two validity comparisons share a block. Bonferroni,
PD, and PL all receive the same pair $(X_q,Y_q)$.


## Square

In [44]:
func_weight = function_weight("Poly", poly_order = 2)
test_kwargs = {
    "alpha": ALPHA,
    "func_weight": func_weight,
    "reference_rng":np.random.default_rng(20260928),
    "max_block_entries": MAX_BLOCK_ENTRIES,
}

bonferroni_null_rejections = np.zeros(
    NUMBER_OF_NULL_PAIRS,
    dtype=bool,
)
bonferroni_null_pvalues = np.full(
    NUMBER_OF_NULL_PAIRS,
    np.nan,
    dtype=float,
)

start_time = time.time()

for pair_index, (block_x, block_y) in enumerate(null_pair_blocks):
    sample_x = decode_simulation(simulation_blocks[null_index][block_x])
    sample_y = decode_simulation(simulation_blocks[null_index][block_y])

    result = bonferroni_test(sample_x, sample_y, **test_kwargs)
    bonferroni_null_rejections[pair_index] = bool(result["reject"])
    bonferroni_null_pvalues[pair_index] = float(
        result.get("adjusted_p_value", np.nan)
    )

    completed = pair_index + 1
    if (
        completed % PROGRESS_EVERY == 0
        or completed == NUMBER_OF_NULL_PAIRS
    ):
        elapsed_minutes = (time.time() - start_time) / 60.0
        current_error = bonferroni_null_rejections[:completed].mean()
        print(
            f"Bonferroni validity {completed}/{NUMBER_OF_NULL_PAIRS} "
            f"completed | type-I error={current_error:.3f} | "
            f"elapsed={elapsed_minutes:.1f} min"
        )
savepkl(bonferroni_null_rejections, RESULT_DIR/'bonferroni_null_square_rejections.pkl')

Bonferroni validity 10/250 completed | type-I error=0.000 | elapsed=1.7 min
Bonferroni validity 20/250 completed | type-I error=0.050 | elapsed=3.4 min
Bonferroni validity 30/250 completed | type-I error=0.067 | elapsed=5.0 min
Bonferroni validity 40/250 completed | type-I error=0.050 | elapsed=6.7 min
Bonferroni validity 50/250 completed | type-I error=0.080 | elapsed=8.4 min
Bonferroni validity 60/250 completed | type-I error=0.067 | elapsed=10.0 min
Bonferroni validity 70/250 completed | type-I error=0.071 | elapsed=11.6 min
Bonferroni validity 80/250 completed | type-I error=0.062 | elapsed=13.2 min
Bonferroni validity 90/250 completed | type-I error=0.056 | elapsed=14.9 min
Bonferroni validity 100/250 completed | type-I error=0.050 | elapsed=16.5 min
Bonferroni validity 110/250 completed | type-I error=0.055 | elapsed=18.1 min
Bonferroni validity 120/250 completed | type-I error=0.050 | elapsed=19.7 min
Bonferroni validity 130/250 completed | type-I error=0.054 | elapsed=21.3 min


## Linear

In [45]:
func_weight = function_weight("Poly", poly_order = 1)
test_kwargs = {
    "alpha": ALPHA,
    "func_weight": func_weight,
    "reference_rng":np.random.default_rng(20260928),
    "max_block_entries": MAX_BLOCK_ENTRIES,
}

bonferroni_null_rejections = np.zeros(
    NUMBER_OF_NULL_PAIRS,
    dtype=bool,
)
bonferroni_null_pvalues = np.full(
    NUMBER_OF_NULL_PAIRS,
    np.nan,
    dtype=float,
)

start_time = time.time()

for pair_index, (block_x, block_y) in enumerate(null_pair_blocks):
    sample_x = decode_simulation(simulation_blocks[null_index][block_x])
    sample_y = decode_simulation(simulation_blocks[null_index][block_y])

    result = bonferroni_test(sample_x, sample_y, **test_kwargs)
    bonferroni_null_rejections[pair_index] = bool(result["reject"])
    bonferroni_null_pvalues[pair_index] = float(
        result.get("adjusted_p_value", np.nan)
    )

    completed = pair_index + 1
    if (
        completed % PROGRESS_EVERY == 0
        or completed == NUMBER_OF_NULL_PAIRS
    ):
        elapsed_minutes = (time.time() - start_time) / 60.0
        current_error = bonferroni_null_rejections[:completed].mean()
        print(
            f"Bonferroni validity {completed}/{NUMBER_OF_NULL_PAIRS} "
            f"completed | type-I error={current_error:.3f} | "
            f"elapsed={elapsed_minutes:.1f} min"
        )
savepkl(bonferroni_null_rejections, RESULT_DIR/'bonferroni_null_linear_rejections.pkl')

Bonferroni validity 10/250 completed | type-I error=0.000 | elapsed=1.6 min
Bonferroni validity 20/250 completed | type-I error=0.050 | elapsed=3.3 min
Bonferroni validity 30/250 completed | type-I error=0.067 | elapsed=4.9 min
Bonferroni validity 40/250 completed | type-I error=0.050 | elapsed=6.5 min
Bonferroni validity 50/250 completed | type-I error=0.080 | elapsed=8.2 min
Bonferroni validity 60/250 completed | type-I error=0.067 | elapsed=9.8 min
Bonferroni validity 70/250 completed | type-I error=0.071 | elapsed=11.4 min
Bonferroni validity 80/250 completed | type-I error=0.062 | elapsed=13.0 min
Bonferroni validity 90/250 completed | type-I error=0.056 | elapsed=14.7 min
Bonferroni validity 100/250 completed | type-I error=0.050 | elapsed=16.3 min
Bonferroni validity 110/250 completed | type-I error=0.055 | elapsed=17.9 min
Bonferroni validity 120/250 completed | type-I error=0.050 | elapsed=19.5 min
Bonferroni validity 130/250 completed | type-I error=0.054 | elapsed=21.2 min
B

## Constant

In [46]:
func_weight = function_weight("constant")
test_kwargs = {
    "alpha": ALPHA,
    "func_weight": func_weight,
    "reference_rng":np.random.default_rng(20260928),
    "max_block_entries": MAX_BLOCK_ENTRIES,
}

bonferroni_null_rejections = np.zeros(
    NUMBER_OF_NULL_PAIRS,
    dtype=bool,
)
bonferroni_null_pvalues = np.full(
    NUMBER_OF_NULL_PAIRS,
    np.nan,
    dtype=float,
)

start_time = time.time()

for pair_index, (block_x, block_y) in enumerate(null_pair_blocks):
    sample_x = decode_simulation(simulation_blocks[null_index][block_x])
    sample_y = decode_simulation(simulation_blocks[null_index][block_y])

    result = bonferroni_test(sample_x, sample_y, **test_kwargs)
    bonferroni_null_rejections[pair_index] = bool(result["reject"])
    bonferroni_null_pvalues[pair_index] = float(
        result.get("adjusted_p_value", np.nan)
    )

    completed = pair_index + 1
    if (
        completed % PROGRESS_EVERY == 0
        or completed == NUMBER_OF_NULL_PAIRS
    ):
        elapsed_minutes = (time.time() - start_time) / 60.0
        current_error = bonferroni_null_rejections[:completed].mean()
        print(
            f"Bonferroni validity {completed}/{NUMBER_OF_NULL_PAIRS} "
            f"completed | type-I error={current_error:.3f} | "
            f"elapsed={elapsed_minutes:.1f} min"
        )

savepkl(bonferroni_null_rejections, RESULT_DIR/'bonferroni_null_constant_rejections.pkl')

Bonferroni validity 10/250 completed | type-I error=0.000 | elapsed=1.6 min
Bonferroni validity 20/250 completed | type-I error=0.000 | elapsed=3.3 min
Bonferroni validity 30/250 completed | type-I error=0.067 | elapsed=4.9 min
Bonferroni validity 40/250 completed | type-I error=0.050 | elapsed=6.6 min
Bonferroni validity 50/250 completed | type-I error=0.040 | elapsed=8.2 min
Bonferroni validity 60/250 completed | type-I error=0.033 | elapsed=9.8 min
Bonferroni validity 70/250 completed | type-I error=0.029 | elapsed=11.5 min
Bonferroni validity 80/250 completed | type-I error=0.037 | elapsed=13.1 min
Bonferroni validity 90/250 completed | type-I error=0.056 | elapsed=14.7 min
Bonferroni validity 100/250 completed | type-I error=0.050 | elapsed=16.4 min
Bonferroni validity 110/250 completed | type-I error=0.045 | elapsed=18.0 min
Bonferroni validity 120/250 completed | type-I error=0.050 | elapsed=19.6 min
Bonferroni validity 130/250 completed | type-I error=0.046 | elapsed=21.2 min
B

In [47]:
pd_null_pvalues = np.full(
    NUMBER_OF_NULL_PAIRS,
    np.nan,
    dtype=float,
)

start_time = time.time()

for pair_index, (block_x, block_y) in enumerate(null_pair_blocks):
    sample_x = decode_simulation(simulation_blocks[null_index][block_x])
    sample_y = decode_simulation(simulation_blocks[null_index][block_y])
    seed_j = int(validity_test_seeds[pair_index])

    _, _, pd_null_pvalues[pair_index] = permutation_test(
        sample_x,
        sample_y,
        num_permutations=B,
        seed=seed_j,
    )

    completed = pair_index + 1
    if (
        completed % PROGRESS_EVERY == 0
        or completed == NUMBER_OF_NULL_PAIRS
    ):
        elapsed_minutes = (time.time() - start_time) / 60.0
        current_error = np.mean(pd_null_pvalues[:completed] <= ALPHA)
        print(
            f"PD validity {completed}/{NUMBER_OF_NULL_PAIRS} completed | "
            f"type-I error={current_error:.3f} | "
            f"elapsed={elapsed_minutes:.1f} min"
        )
savepkl(pd_null_pvalues, RESULT_DIR/'pd_null_pvalues.pkl')

PD validity 10/250 completed | type-I error=0.000 | elapsed=0.9 min
PD validity 20/250 completed | type-I error=0.100 | elapsed=1.9 min
PD validity 30/250 completed | type-I error=0.067 | elapsed=2.9 min
PD validity 40/250 completed | type-I error=0.050 | elapsed=3.8 min
PD validity 50/250 completed | type-I error=0.040 | elapsed=4.8 min
PD validity 60/250 completed | type-I error=0.050 | elapsed=5.7 min
PD validity 70/250 completed | type-I error=0.043 | elapsed=6.7 min
PD validity 80/250 completed | type-I error=0.050 | elapsed=7.7 min
PD validity 90/250 completed | type-I error=0.067 | elapsed=8.6 min
PD validity 100/250 completed | type-I error=0.060 | elapsed=9.6 min
PD validity 110/250 completed | type-I error=0.055 | elapsed=10.5 min
PD validity 120/250 completed | type-I error=0.050 | elapsed=11.5 min
PD validity 130/250 completed | type-I error=0.054 | elapsed=12.4 min
PD validity 140/250 completed | type-I error=0.050 | elapsed=13.4 min
PD validity 150/250 completed | type-I 

In [48]:
pl_null_pvalues = np.full(
    NUMBER_OF_NULL_PAIRS,
    np.nan,
    dtype=float,
)

first_null_stream = iter_pl_cache(RADIUS_SHIFTS[null_index])
second_null_stream = iter_pl_cache(RADIUS_SHIFTS[null_index])
first_half = itertools.islice(
    first_null_stream,
    0,
    NUMBER_OF_NULL_PAIRS,
)
second_half = itertools.islice(
    second_null_stream,
    NUMBER_OF_NULL_PAIRS,
    NSET,
)

start_time = time.time()
completed = 0

try:
    for pair_index, (sample_x, sample_y) in enumerate(
        zip(first_half, second_half)
    ):
        seed_j = int(validity_test_seeds[pair_index])

        pl_null_pvalues[pair_index] = permutation_pl_test_fast(
            sample_x,
            sample_y,
            seed=seed_j,
            num_perms=B,
        )

        completed = pair_index + 1
        if (
            completed % PROGRESS_EVERY == 0
            or completed == NUMBER_OF_NULL_PAIRS
        ):
            elapsed_minutes = (time.time() - start_time) / 60.0
            current_error = np.mean(
                pl_null_pvalues[:completed] <= ALPHA
            )
            print(
                f"PL validity {completed}/{NUMBER_OF_NULL_PAIRS} "
                f"completed | type-I error={current_error:.3f} | "
                f"elapsed={elapsed_minutes:.1f} min"
            )
finally:
    first_null_stream.close()
    second_null_stream.close()

if completed != NUMBER_OF_NULL_PAIRS:
    raise RuntimeError(
        f"Expected {NUMBER_OF_NULL_PAIRS} PL validity comparisons, "
        f"but completed {completed}."
    )
savepkl(pl_null_pvalues, RESULT_DIR/'pl_null_pvalues.pkl')

PL validity 10/250 completed | type-I error=0.000 | elapsed=0.0 min
PL validity 20/250 completed | type-I error=0.050 | elapsed=0.1 min
PL validity 30/250 completed | type-I error=0.033 | elapsed=0.1 min
PL validity 40/250 completed | type-I error=0.050 | elapsed=0.1 min
PL validity 50/250 completed | type-I error=0.060 | elapsed=0.1 min
PL validity 60/250 completed | type-I error=0.067 | elapsed=0.1 min
PL validity 70/250 completed | type-I error=0.057 | elapsed=0.2 min
PL validity 80/250 completed | type-I error=0.050 | elapsed=0.2 min
PL validity 90/250 completed | type-I error=0.056 | elapsed=0.2 min
PL validity 100/250 completed | type-I error=0.060 | elapsed=0.2 min
PL validity 110/250 completed | type-I error=0.055 | elapsed=0.3 min
PL validity 120/250 completed | type-I error=0.058 | elapsed=0.3 min
PL validity 130/250 completed | type-I error=0.054 | elapsed=0.3 min
PL validity 140/250 completed | type-I error=0.050 | elapsed=0.3 min
PL validity 150/250 completed | type-I erro

In [51]:
POWER_SAMPLE_SIZES = (15, 25, 40)
POWER_WEIGHT_SPECS = (
    ("Bonferroni (square)", function_weight("Poly", poly_order=2)),
    ("Bonferroni (linear)", function_weight("Poly", poly_order=1)),
    ("Bonferroni (constant)", function_weight("constant")),
)
POWER_METHODS = [name for name, _ in POWER_WEIGHT_SPECS] + ["PD", "PL"]

if SAMPLE_SIZE != 50 or max(POWER_SAMPLE_SIZES) > SAMPLE_SIZE:
    raise ValueError("The existing PL caches must contain 50 diagrams per block.")


def power_path(n, method):
    return RESULT_DIR / f"torus_radius_power_n{n}_{method}.pkl"


def power_progress(path, shape):
    if path.exists():
        result = np.asarray(loadpkl(path), dtype=float)
        if result.shape != shape:
            raise ValueError(f"Wrong checkpoint shape in {path}: {result.shape}")
        return result
    return np.full(shape, np.nan)


def save_power_progress(result, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    savepkl(result, temporary)
    os.replace(temporary, path)


def decode_power_sample(raw_block, n):
    return [decode_h1_diagram(item) for item in selected_raw_diagrams(raw_block)[:n]]


In [56]:
# Bonferroni power for the three weights; no n=50 recomputation.
for n in [40]:#POWER_SAMPLE_SIZES:
    path = power_path(n, "bonferroni_rejections")
    reject = power_progress(
        path, (len(POWER_WEIGHT_SPECS), len(alternative_indices), NSET)
    )
    start = time.time()

    for replication in range(NSET):
        if np.isfinite(reject[:, :, replication]).all():
            continue
        null_sample = decode_power_sample(
            simulation_blocks[null_index][replication], n
        )
        for output_index, shift_index in enumerate(alternative_indices):
            shifted_sample = decode_power_sample(
                simulation_blocks[shift_index][replication], n
            )
            for weight_index, (_, weight) in enumerate(POWER_WEIGHT_SPECS):
                if np.isfinite(reject[weight_index, output_index, replication]):
                    continue
                # Each comparison gets a reproducible reference sample.
                reference_rng = np.random.default_rng(
                    np.random.SeedSequence([20260928, n, output_index, replication])
                )
                result = bonferroni_test(
                    null_sample, shifted_sample,
                    alpha=ALPHA, func_weight=weight,
                    reference_rng=reference_rng,
                    max_block_entries=MAX_BLOCK_ENTRIES,
                )
                reject[weight_index, output_index, replication] = int(result["reject"])

        completed = replication + 1
        if completed % 25 == 0 or completed == NSET:
            save_power_progress(reject, path)
        if completed % 50 == 0 or completed == NSET:
            print(f"Bonferroni n=m={n}: {completed}/{NSET}, "
                  f"elapsed={(time.time() - start) / 60:.1f} min")

    if not np.isfinite(reject).all():
        raise RuntimeError(f"Bonferroni n=m={n} did not finish.")
    save_power_progress(reject, path)
    print(f"Saved: {path}")


Bonferroni n=m=40: 50/500, elapsed=31.6 min
Bonferroni n=m=40: 100/500, elapsed=94.2 min
Bonferroni n=m=40: 150/500, elapsed=156.8 min
Bonferroni n=m=40: 200/500, elapsed=219.5 min
Bonferroni n=m=40: 250/500, elapsed=282.0 min
Bonferroni n=m=40: 300/500, elapsed=344.6 min
Bonferroni n=m=40: 350/500, elapsed=406.9 min
Bonferroni n=m=40: 400/500, elapsed=469.2 min
Bonferroni n=m=40: 450/500, elapsed=531.5 min
Bonferroni n=m=40: 500/500, elapsed=593.8 min
Saved: ../../results/simulation_results/Torus_results/torus_radius_power_n40_bonferroni_rejections.pkl


In [57]:
# PD permutation power, with the same per-comparison seeds as the n=50 run.
for n in POWER_SAMPLE_SIZES:
    path = power_path(n, "pd_pvalues")
    pvalues = power_progress(path, (len(alternative_indices), NSET))
    start = time.time()

    for replication in range(NSET):
        if np.isfinite(pvalues[:, replication]).all():
            continue
        null_sample = decode_power_sample(
            simulation_blocks[null_index][replication], n
        )
        for output_index, shift_index in enumerate(alternative_indices):
            if np.isfinite(pvalues[output_index, replication]):
                continue
            shifted_sample = decode_power_sample(
                simulation_blocks[shift_index][replication], n
            )
            seed_j = int(power_test_seeds[output_index, replication])
            _, _, pvalues[output_index, replication] = permutation_test(
                null_sample, shifted_sample, num_permutations=B, seed=seed_j
            )

        completed = replication + 1
        if completed % 25 == 0 or completed == NSET:
            save_power_progress(pvalues, path)
        if completed % 50 == 0 or completed == NSET:
            print(f"PD n=m={n}: {completed}/{NSET}, "
                  f"elapsed={(time.time() - start) / 60:.1f} min")

    if not np.isfinite(pvalues).all():
        raise RuntimeError(f"PD n=m={n} did not finish.")
    save_power_progress(pvalues, path)
    print(f"Saved: {path}")


PD n=m=15: 50/500, elapsed=1.8 min
PD n=m=15: 100/500, elapsed=3.5 min
PD n=m=15: 150/500, elapsed=5.3 min
PD n=m=15: 200/500, elapsed=7.1 min
PD n=m=15: 250/500, elapsed=8.9 min
PD n=m=15: 300/500, elapsed=10.6 min
PD n=m=15: 350/500, elapsed=12.4 min
PD n=m=15: 400/500, elapsed=14.2 min
PD n=m=15: 450/500, elapsed=15.9 min
PD n=m=15: 500/500, elapsed=17.7 min
Saved: ../../results/simulation_results/Torus_results/torus_radius_power_n15_pd_pvalues.pkl
PD n=m=25: 50/500, elapsed=4.8 min
PD n=m=25: 100/500, elapsed=9.7 min
PD n=m=25: 150/500, elapsed=14.5 min
PD n=m=25: 200/500, elapsed=19.4 min
PD n=m=25: 250/500, elapsed=24.2 min
PD n=m=25: 300/500, elapsed=29.1 min
PD n=m=25: 350/500, elapsed=33.9 min
PD n=m=25: 400/500, elapsed=38.7 min
PD n=m=25: 450/500, elapsed=43.6 min
PD n=m=25: 500/500, elapsed=48.4 min
Saved: ../../results/simulation_results/Torus_results/torus_radius_power_n25_pd_pvalues.pkl
PD n=m=40: 50/500, elapsed=12.4 min
PD n=m=40: 100/500, elapsed=24.7 min
PD n=m=40: 1

In [58]:
# PL power: use the first n landscapes from each existing 50-diagram cache block.
for n in POWER_SAMPLE_SIZES:
    path = power_path(n, "pl_pvalues")
    pvalues = power_progress(path, (len(alternative_indices), NSET))
    start = time.time()

    for output_index, shift_index in enumerate(alternative_indices):
        if np.isfinite(pvalues[output_index]).all():
            continue
        null_stream = iter_pl_cache(RADIUS_SHIFTS[null_index])
        shifted_stream = iter_pl_cache(RADIUS_SHIFTS[shift_index])
        try:
            for replication, (null_block, shifted_block) in enumerate(
                zip(null_stream, shifted_stream)
            ):
                if np.isfinite(pvalues[output_index, replication]):
                    continue
                seed_j = int(power_test_seeds[output_index, replication])
                pvalues[output_index, replication] = permutation_pl_test_fast(
                    null_block[:n], shifted_block[:n], seed=seed_j, num_perms=B
                )
                completed = replication + 1
                if completed % 25 == 0 or completed == NSET:
                    save_power_progress(pvalues, path)
        finally:
            null_stream.close()
            shifted_stream.close()
        print(f"PL n=m={n}, shift={RADIUS_SHIFTS[shift_index]:.3f}: "
              f"done, elapsed={(time.time() - start) / 60:.1f} min")

    if not np.isfinite(pvalues).all():
        raise RuntimeError(f"PL n=m={n} did not finish.")
    save_power_progress(pvalues, path)
    print(f"Saved: {path}")


PL n=m=15, shift=0.010: done, elapsed=0.2 min
PL n=m=15, shift=0.016: done, elapsed=0.3 min
PL n=m=15, shift=0.026: done, elapsed=0.5 min
PL n=m=15, shift=0.036: done, elapsed=0.6 min
Saved: ../../results/simulation_results/Torus_results/torus_radius_power_n15_pl_pvalues.pkl
PL n=m=25, shift=0.010: done, elapsed=0.2 min
PL n=m=25, shift=0.016: done, elapsed=0.4 min
PL n=m=25, shift=0.026: done, elapsed=0.7 min
PL n=m=25, shift=0.036: done, elapsed=0.9 min
Saved: ../../results/simulation_results/Torus_results/torus_radius_power_n25_pl_pvalues.pkl
PL n=m=40, shift=0.010: done, elapsed=0.3 min
PL n=m=40, shift=0.016: done, elapsed=0.6 min
PL n=m=40, shift=0.026: done, elapsed=1.0 min
PL n=m=40, shift=0.036: done, elapsed=1.3 min
Saved: ../../results/simulation_results/Torus_results/torus_radius_power_n40_pl_pvalues.pkl
